In [ ]:
# ==============================================================================
# Part 1: Imports & Setup
# ==============================================================================

!pip install torch torchvision numpy scipy scikit-learn matplotlib pillow tensorboard

# 1. Standard Library Imports
import argparse
import math
import os
from pprint import pprint
import random
import shutil
import time

# 2. Third-Party Scientific & ML Libraries
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import scipy as sp
import scipy.stats
from scipy.optimize import linear_sum_assignment
from sklearn import ensemble, linear_model, metrics

# 3. PyTorch Core, Distributions & Neural Network Modules
import torch
from torch import autograd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.parameter import Parameter
from torch.distributions.normal import Normal
from torch.distributions.multivariate_normal import MultivariateNormal
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter

# 4. Torchvision Imports
import torchvision
from torchvision import datasets, transforms
from torchvision.utils import save_image

# 5. Global Device Configuration & Determinism Setup
cuda = torch.cuda.is_available()
device = torch.device("cuda" if cuda else "cpu")


  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-manylinux_2_31_x86_64.whl.metadata (1.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 2.2 MB/s eta 0:00:00 MB/s eta 0:00:01:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 30.3 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 28.0 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 29.1 MB/s eta 0:00:00m eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 28.0 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 30.2 MB/s eta 0:00:000:00:01m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 28.7 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 32.2 MB/s eta 0:00:00MB/s eta 0:00:01
   ━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# ==============================================================================
# Part 2: Statistical Functions, Priors, Datasets, and Metric Helpers
# ==============================================================================

# Base loss definition
bce = torch.nn.BCEWithLogitsLoss(reduction="none")


# ------------------------------------------------------------------------------
# Parameter Extraction & Sampling
# ------------------------------------------------------------------------------

def gaussian_parameters(h, dim=-1):
    """Splits tensor into mean and softplus-constrained variance."""
    m, h = torch.split(h, h.size(dim) // 2, dim=dim)
    v = F.softplus(h) + 1e-8
    return m, v


def sample_gaussian(m, v):
    """Reparameterization trick sampling: z = mu + std * eps."""
    sample = torch.randn(m.shape, device=m.device)
    return m + (v ** 0.5) * sample


# ------------------------------------------------------------------------------
# Log Probability and Divergence Calculations
# ------------------------------------------------------------------------------

def log_normal(x, m, var):
    """Computes log-density of a diagonal Gaussian distribution."""
    const = -0.5 * x.size(-1) * torch.log(2 * torch.tensor(np.pi, device=x.device))
    log_det = -0.5 * torch.sum(torch.log(var + 1e-8), dim=-1)
    log_exp = -0.5 * torch.sum((x - m) ** 2 / (var + 1e-8), dim=-1)
    return const + log_det + log_exp


def log_bernoulli_with_logits(x, logits):
    """Computes log Bernoulli likelihood summed over all dimensions except batch."""
    loss = bce(input=logits.view(logits.size(0), -1), target=x.view(x.size(0), -1))
    return -loss.sum(dim=-1)


def log_prob(qm, qv, pm, pv):
    """
    Computes Gaussian KL Divergence: KL( q(z) || p(z) )
    Parameters (qm, pm) are means; (qv, pv) are strict VARIANCES (sigma^2).
    """
    element_wise = 0.5 * (
        torch.log((pv + 1e-8) / (qv + 1e-8)) + (qv + (qm - pm).pow(2)) / (pv + 1e-8) - 1.0
    )
    return element_wise.sum(-1)


# ------------------------------------------------------------------------------
# Priors & Conditional Transformations (Vectorized)
# ------------------------------------------------------------------------------

def condition_prior(scale, label, dim):
    """Constructs normalized conditional Gaussian prior (mean, var) from labels."""
    scale_tensor = torch.as_tensor(scale, dtype=torch.float32, device=label.device)
    # Normalized label per concept
    norm_label = (label - scale_tensor[:, 0]) / scale_tensor[:, 1]
    # Expand to shape: (batch_size, num_concepts, dim)
    mean = norm_label.unsqueeze(-1).expand(-1, -1, dim).clone()
    var = torch.ones_like(mean)
    return mean, var


# ------------------------------------------------------------------------------
# Flow Datasets & DataLoaders
# ------------------------------------------------------------------------------

class SyntheticLabeled(Dataset):
    """Dataset loader for synthetic causal flow images with metadata labels."""
    def __init__(self, root, dataset="train"):
        root = os.path.join(root, dataset)
        self.dataset = dataset

        if os.path.exists(root):
            imgs = sorted([f for f in os.listdir(root) if f.endswith(".png") or f.endswith(".jpg")])
            self.imgs = [os.path.join(root, k) for k in imgs]
            # Fixed: Cast to float to handle float filenames from Part 5
            self.imglabel = [list(map(float, k[:-4].split("_")[1:])) for k in imgs]
            label = np.asarray(self.imglabel)
            self.label_avg = np.mean(label, axis=0) if len(label) > 0 else 0
            self.label_std = np.std(label, axis=0) if len(label) > 0 else 1
        else:
            self.imgs = []
            self.imglabel = []
            self.label_avg = 0
            self.label_std = 1

        # Fixed: Explicitly resize to ensure 96x96 dimensions for linear layers
        self.transforms = transforms.Compose([
            transforms.Resize((96, 96)),
            transforms.ToTensor()  # Converts [0, 255] PIL image to [0.0, 1.0] Tensor
        ])

    def __getitem__(self, idx):
        img_path = self.imgs[idx]
        label = torch.from_numpy(np.asarray(self.imglabel[idx])).float()
        pil_img = Image.open(img_path).convert("RGBA")  # Ensures fixed 4-channel image
        
        data = self.transforms(pil_img)
        return data, label

    def __len__(self):
        return len(self.imgs)


def get_batch_unin_dataset_withlabel(dataset_dir, batch_size, dataset="train"):
    """Instantiates DataLoader for SyntheticLabeled dataset."""
    dataset_inst = SyntheticLabeled(dataset_dir, dataset=dataset)
    print(f"[*] Loaded '{dataset}' split with {len(dataset_inst)} samples.")
    dataloader = DataLoader(
        dataset_inst, 
        batch_size=batch_size, 
        shuffle=(dataset == "train"), 
        drop_last=False
    )
    return dataloader


# ------------------------------------------------------------------------------
# Checkpointing Utilities
# ------------------------------------------------------------------------------

def load_model_by_name(model, global_step):
    file_path = os.path.join(
        "checkpoints", model.name, "model-{:05d}.pt".format(global_step)
    )
    state = torch.load(file_path, map_location="cpu")
    model.load_state_dict(state)
    print("Loaded from {}".format(file_path))


def save_model_by_name(model, global_step):
    save_dir = os.path.join("checkpoints", model.name)
    if not os.path.exists(save_dir):
        os.makedirs(save_dir, exist_ok=True)
    file_path = os.path.join(save_dir, "model-{:05d}.pt".format(global_step))
    torch.save(model.state_dict(), file_path)
    print("Saved to {}".format(file_path))


# ------------------------------------------------------------------------------
# IRS Metric (Interventional Robustness Score)
# ------------------------------------------------------------------------------

def scalable_disentanglement_score(gen_factors, latents, diff_quantile=0.99):
    """Computes IRS scores for generative factors against latent variables."""
    num_gen = gen_factors.shape[1]
    num_lat = latents.shape[1]

    max_deviations = np.max(np.abs(latents - latents.mean(axis=0)), axis=0)
    cum_deviations = np.zeros([num_lat, num_gen])

    for i in range(num_gen):
        unique_factors = np.unique(gen_factors[:, i], axis=0)
        num_distinct_factors = unique_factors.shape[0]
        for k in range(num_distinct_factors):
            match = gen_factors[:, i] == unique_factors[k]
            e_loc = np.mean(latents[match, :], axis=0)

            diffs = np.abs(latents[match, :] - e_loc)
            max_diffs = np.percentile(diffs, q=diff_quantile * 100, axis=0)
            cum_deviations[:, i] += max_diffs

        cum_deviations[:, i] /= num_distinct_factors

    normalized_deviations = cum_deviations / (max_deviations[:, np.newaxis] + 1e-12)
    irs_matrix = 1.0 - normalized_deviations
    disentanglement_scores = irs_matrix.max(axis=1)

    if np.sum(max_deviations) > 0.0:
        avg_score = np.average(disentanglement_scores, weights=max_deviations)
    else:
        avg_score = np.mean(disentanglement_scores)

    return {
        "disentanglement_scores": disentanglement_scores,
        "avg_score": avg_score,
        "parents": irs_matrix.argmax(axis=1),
        "IRS_matrix": irs_matrix,
        "max_deviations": max_deviations,
    }


def compute_irs(rep, y, diff_quantile=0.99):
    """Computes the summary IRS dictionary."""
    if not rep.any():
        irs_score = 0.0
    else:
        irs_score = scalable_disentanglement_score(
            y.T, rep.T, diff_quantile=diff_quantile
        )["avg_score"]

    return {"IRS": irs_score, "num_active_dims": np.sum(rep)}


# ------------------------------------------------------------------------------
# DCI Metric (Disentanglement, Completeness, Informativeness)
# ------------------------------------------------------------------------------

def compute_importance_gbt(x_train, y_train, x_test, y_test):
    """Computes feature importance matrix via Gradient Boosted Trees."""
    num_factors = y_train.shape[0]
    num_codes = x_train.shape[0]
    importance_matrix = np.zeros(
        shape=[num_codes, num_factors], dtype=np.float64
    )
    train_loss = []
    test_loss = []

    for i in range(num_factors):
        model = ensemble.GradientBoostingRegressor()
        model.fit(x_train.T, y_train[i, :])
        importance_matrix[:, i] = np.abs(model.feature_importances_)
        train_loss.append(
            metrics.mean_squared_error(y_train[i, :], model.predict(x_train.T))
        )
        test_loss.append(
            metrics.mean_squared_error(y_test[i, :], model.predict(x_test.T))
        )

    return importance_matrix, np.mean(train_loss), np.mean(test_loss)


def disentanglement_per_code(importance_matrix):
    row_sums = importance_matrix.sum(axis=1, keepdims=True) + 1e-11
    norm_matrix = importance_matrix / row_sums
    num_factors = max(importance_matrix.shape[1], 2)
    return 1.0 - scipy.stats.entropy(norm_matrix.T + 1e-11, base=num_factors)


def disentanglement(importance_matrix):
    per_code = disentanglement_per_code(importance_matrix)
    total_sum = importance_matrix.sum()
    if total_sum == 0.0:
        importance_matrix = np.ones_like(importance_matrix)
        total_sum = importance_matrix.sum()
    code_importance = importance_matrix.sum(axis=1) / total_sum
    return np.sum(per_code * code_importance), code_importance


def completeness_per_factor(importance_matrix):
    col_sums = importance_matrix.sum(axis=0, keepdims=True) + 1e-11
    norm_matrix = importance_matrix / col_sums
    num_codes = max(importance_matrix.shape[0], 2)
    return 1.0 - scipy.stats.entropy(norm_matrix + 1e-11, base=num_codes)


def completeness(importance_matrix):
    per_factor = completeness_per_factor(importance_matrix)
    total_sum = importance_matrix.sum()
    if total_sum == 0.0:
        importance_matrix = np.ones_like(importance_matrix)
        total_sum = importance_matrix.sum()
    factor_importance = importance_matrix.sum(axis=0) / total_sum
    return np.sum(per_factor * factor_importance)


def _compute_dci(mus_train, ys_train, mus_test, ys_test):
    """Computes overall DCI metric profile."""
    importance_matrix, train_err, test_err = compute_importance_gbt(
        mus_train, ys_train, mus_test, ys_test
    )
    disent, code_importance = disentanglement(importance_matrix)
    comp = completeness(importance_matrix)

    scores = {
        "informativeness_train": train_err,
        "informativeness_test": test_err,
        "disentanglement": disent,
        "completeness": comp,
    }
    return scores, importance_matrix, code_importance


In [ ]:
# ==============================================================================
# Part 3: Causal Normalizing Flows & Structural Causal Model (SCM) Layers
# ==============================================================================



class MLP(nn.Module):
    """Masked MLP condition network for Autoregressive/Causal Flow transforms."""

    def __init__(self, nin, nout, nh=100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(nin, nh),
            nn.ReLU(),
            nn.Linear(nh, nh),
            nn.ReLU(),
            nn.Linear(nh, nout),
        )

    def forward(self, x, mask):
        # Apply binary structural adjacency mask
        return self.net(x * mask)


# ------------------------------------------------------------------------------
# Multivariate Causal Normalizing Flow
# ------------------------------------------------------------------------------


class PriorMultivariateCausalFlow(nn.Module):
    """Autoregressive Causal Normalizing Flow parameterized with structural adjacency."""

    def __init__(
        self, dim, k, C=None, net_class=MLP, nh=100, scale=True, shift=True
    ):
        super().__init__()
        self.dim = dim
        self.k = k
        self.register_buffer(
            "C",
            torch.tensor(C, dtype=torch.float32)
            if not isinstance(C, torch.Tensor)
            else C.float(),
        )

        total_dim = self.dim * self.k
        self.scale = scale
        self.shift = shift

        if scale:
            self.s_cond = net_class(total_dim, self.k, nh)
        if shift:
            self.t_cond = net_class(total_dim, self.k, nh)

    def forward(self, e, latent=None, target=None, value=None):
        batch_size = e.shape[0]
        total_dims = self.dim * self.k
        z = torch.zeros(batch_size, self.dim, self.k, device=e.device)
        log_det = torch.zeros(batch_size, device=e.device)

        cond_source = latent if latent is not None else z

        for i in range(self.dim):
            if (self.C[:, i] == 1).any() and (target != i):
                mask = (
                    self.C[:, i]
                    .repeat_interleave(self.k)
                    .to(device=e.device, dtype=torch.float32)
                )
            else:
                mask = torch.zeros(total_dims, device=e.device)

            cond_flat = (
                cond_source.view(batch_size, total_dims)
                if latent is not None
                else z.view(batch_size, total_dims)
            )
            s = (
                self.s_cond(cond_flat, mask)
                if self.scale
                else torch.zeros(batch_size, self.k, device=e.device)
            )
            t = (
                self.t_cond(cond_flat, mask)
                if self.shift
                else torch.zeros(batch_size, self.k, device=e.device)
            )

            z_i = torch.exp(s) * e[:, i, :] + t
            if target is not None and value is not None and target == i:
                z_i = value

            z = z.clone()
            z[:, i, :] = z_i
            log_det = log_det + torch.sum(s, dim=1)

        return z, log_det

    def backward(self, z, target=None, value=None):
        batch_size = z.shape[0]
        total_dims = self.dim * self.k
        e = torch.zeros(batch_size, self.dim, self.k, device=z.device)
        log_det = torch.zeros(batch_size, device=z.device)

        for i in range(self.dim):
            if (self.C[:, i] == 1).any() and (target != i):
                mask = (
                    self.C[:, i]
                    .repeat_interleave(self.k)
                    .to(device=z.device, dtype=torch.float32)
                )
            else:
                mask = torch.zeros(total_dims, device=z.device)

            z_flat = z.view(batch_size, total_dims)
            s = (
                self.s_cond(z_flat, mask)
                if self.scale
                else torch.zeros(batch_size, self.k, device=z.device)
            )
            t = (
                self.t_cond(z_flat, mask)
                if self.shift
                else torch.zeros(batch_size, self.k, device=z.device)
            )

            e_i = torch.exp(-s) * (z[:, i, :] - t)
            e = e.clone()
            e[:, i, :] = e_i
            log_det = log_det - torch.sum(s, dim=1)

        return e, log_det


In [ ]:
# ==============================================================================
# Part 4: Encoders, Decoders & ICM_VAE Architecture
# ==============================================================================

# ------------------------------------------------------------------------------
# Base MLP Encoders & Decoders
# ------------------------------------------------------------------------------

class Encoder(nn.Module):
    """Gaussian MLP Encoder parameterized for continuous latent variables."""

    def __init__(self, z_dim, channel=4, y_dim=0):
        super().__init__()
        self.z_dim = z_dim
        self.y_dim = y_dim
        self.channel = channel

        self.net = nn.Sequential(
            nn.Linear(self.channel * 96 * 96 + y_dim, 900),
            nn.ELU(),
            nn.Linear(900, 300),
            nn.ELU(),
            nn.Linear(300, 2 * z_dim),
        )

    def encode(self, x, y=None):
        x_flat = x.view(x.size(0), -1)
        xy = x_flat if y is None else torch.cat((x_flat, y.view(y.size(0), -1)), dim=1)
        h = self.net(xy)
        return gaussian_parameters(h, dim=1)


class Decoder(nn.Module):
    """Standard MLP Decoder mapping latent vectors back to image space."""

    def __init__(self, z_dim, channel=4, y_dim=0):
        super().__init__()
        self.z_dim = z_dim
        self.y_dim = y_dim
        self.channel = channel
        self.net = nn.Sequential(
            nn.Linear(z_dim + y_dim, 300),
            nn.ELU(),
            nn.Linear(300, 300),
            nn.ELU(),
            nn.Linear(300, self.channel * 96 * 96),
        )

    def decode(self, z, y=None):
        zy = z if y is None else torch.cat((z, y), dim=1)
        return self.net(zy)


class Decoder_DAG(nn.Module):
    """Concept-factorized MLP Decoder evaluating distinct generative streams."""

    def __init__(self, z_dim, concept=4, z1_dim=4, channel=4, y_dim=0):
        super().__init__()
        self.z_dim = z_dim
        self.concept = concept
        self.z1_dim = z1_dim  # Dimension per concept
        self.y_dim = y_dim
        self.channel = channel

        def make_stream():
            return nn.Sequential(
                nn.Linear(self.z1_dim + self.y_dim, 300),
                nn.ELU(),
                nn.Linear(300, 300),
                nn.ELU(),
                nn.Linear(300, 1024),
                nn.ELU(),
                nn.Linear(1024, self.channel * 96 * 96),
            )

        self.streams = nn.ModuleList(
            [make_stream() for _ in range(self.concept)]
        )

        self.net_full = nn.Sequential(
            nn.Linear(self.z_dim, 300),
            nn.ELU(),
            nn.Linear(300, 300),
            nn.ELU(),
            nn.Linear(300, 1024),
            nn.ELU(),
            nn.Linear(1024, 1024),
            nn.ELU(),
            nn.Linear(1024, self.channel * 96 * 96),
        )

    def decode(self, z, u=None, y=None):
        z = z.view(-1, self.z_dim)
        h = self.net_full(z)
        return h, h, h, h, h

    def decode_sep(self, z, u=None, y=None):
        batch_size = z.size(0)
        z = z.view(batch_size, self.concept, self.z1_dim)

        rx = []
        for i in range(self.concept):
            zi = z[:, i, :]
            if y is not None:
                zi = torch.cat([zi, y.view(batch_size, -1)], dim=1)
            rx.append(self.streams[i](zi))

        h = sum(rx) / self.concept
        return h, h, h, h, h


# ------------------------------------------------------------------------------
# Full ICM-VAE Module
# ------------------------------------------------------------------------------

class ICM_VAE(nn.Module):

    def __init__(
        self,
        name="icm_vae_cdp",
        dataset="synthetic",
        z_dim=16,
        z1_dim=4,  # num_concepts
        z2_dim=4,  # dim_per_concept
        C=None,
        scale=None,
        alpha=0.1,
        beta=1.0,
    ):
        super().__init__()
        self.name = name
        self.z_dim = z_dim
        self.z1_dim = z1_dim  # concepts
        self.z2_dim = z2_dim  # k (dim per concept)
        self.channel = 4
        self.scale = scale
        self.beta = beta
        self.alpha = alpha
        self.C = C

        # Submodules
        self.enc = Encoder(
            z_dim=self.z_dim, channel=self.channel, y_dim=0
        )
        self.dec = Decoder_DAG(
            z_dim=self.z_dim,
            concept=self.z1_dim,
            z1_dim=self.z2_dim,
            channel=self.channel,
        )

        # Causal Normalizing Flow module from Part 3
        self.causal_flow = PriorMultivariateCausalFlow(
            dim=self.z1_dim, k=self.z2_dim, C=self.C
        )
        self.prior = None

    def forward(
        self,
        x,
        label,
        mask=None,
        traversal=None,
        value=None,
        sample=False,
        lambdav=0.001,
    ):
        batch_size = x.size(0)
        device = x.device

        # 1. Encode
        eps_m, eps_v = self.enc.encode(x)
        eps_m = eps_m.reshape(batch_size, self.z1_dim, self.z2_dim)

        # 2. Causal Normalizing Flow Forward / Interventions
        if mask is not None:
            z_m, log_det_z = self.causal_flow(eps_m, target=mask, value=value)
            z_m[:, 3, :] = torch.abs(z_m[:, 3, :])
        elif traversal is not None:
            z_m, log_det_z = self.causal_flow(eps_m)
            z_m[:, traversal, :] = value
        else:
            z_m, log_det_z = self.causal_flow(eps_m)

        z_v = torch.zeros_like(z_m)

        # 3. Sampling Latents (Reparameterization Trick)
        z_given_dag = (
            sample_gaussian(z_m, z_v * lambdav + 1e-8)
            if sample
            else z_m
        )

        # 4. Decode
        x_hat, *_ = self.dec.decode_sep(
            z_given_dag.reshape(batch_size, self.z_dim)
        )

        # 5. Reconstruction Loss (Bernoulli log-likelihood)
        rec = -torch.mean(log_bernoulli_with_logits(x, x_hat.reshape_as(x)))

        # 6. Prior & KL Divergence
        cp_m, cp_v = condition_prior(self.scale, label, self.z2_dim)

        if self.prior is not None:
            cp_m = self.prior(label.to(device), cp_m.to(device), z=z_m)

        # Prior and Posterior Variances (Unit variance = 1.0)
        cp_v = torch.ones(batch_size, self.z1_dim, self.z2_dim, device=device)
        p_m = torch.zeros(batch_size, self.z_dim, device=device)
        p_v = torch.ones(batch_size, self.z_dim, device=device)
        eps_v_flat = torch.ones(batch_size, self.z_dim, device=device)

        kl_fn = log_prob

        # Base noise KL with Jacobian log-determinant adjustment
        base_kl = kl_fn(
            eps_m.view(batch_size, -1),
            eps_v_flat,
            p_m,
            p_v,
        ) - log_det_z
        kl = self.alpha * base_kl

        # Concept-wise Prior Matching KL
        z_posterior_var = torch.ones(batch_size, self.z2_dim, device=device)
        for i in range(self.z1_dim):
            kl = kl + self.beta * kl_fn(
                z_m[:, i, :], z_posterior_var, cp_m[:, i, :], cp_v[:, i, :]
            )

        kl = torch.mean(kl)
        neg_elbo = rec + kl

        return (
            neg_elbo,
            kl,
            rec,
            x_hat.reshape_as(x),
            z_given_dag,
            cp_m,
        )


In [ ]:
# ============================================================================
# Part 5: Synthetic Causal Water-Flow Dataset Generation
# ============================================================================

import matplotlib
matplotlib.use('Agg')  # Fast headless backend
import matplotlib.pyplot as plt


def generate_flow_dataset(
    output_dir: str = './data/flow_noise',
    test_split_ratio: int = 5,  # 1 in every 5 images goes to test (20%)
    seed: int = 42
):
    """
    Generates synthetic causal images of a cup leaking fluid with causal dependencies:
    - ball_r -> water_height (h)
    - h, deep -> water_trajectory (x_true)
    """
    np.random.seed(seed)

    train_dir = os.path.join(output_dir, 'train')
    test_dir = os.path.join(output_dir, 'test')
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    print(f"Generating synthetic causal dataset at '{output_dir}'...")

    total_samples = 30 * 30 * 9  # 8,100 images
    count = 0
    generated_count = 0
    start_time = time.time()

    plt.ioff()

    for r in range(5, 35):
        ball_r = r / 30.0
        for h_raw in range(10, 40):
            h = pow(ball_r, 3) + (h_raw / 10.0)
            for hole in range(6, 15):
                deep = hole / 3.0

                fig, ax = plt.subplots(figsize=(1.0, 1.0), dpi=96)

                # 1. Water in cup
                rect = plt.Rectangle((3.0, 0.0), 5.0, 5.0 + h, color='lightskyblue')
                ax.add_patch(rect)

                # 2. Submerged ball
                ball = plt.Circle((5.5, ball_r + 0.5), ball_r, color='firebrick', zorder=3)
                ax.add_patch(ball)

                # 3. Cup boundaries (polygons)
                left = plt.Polygon([[3.0, 0.0], [3.0, 19.0]], color='black', linewidth=2)
                right_1 = plt.Polygon([[8.0, 0.0], [8.0, deep]], color='black', linewidth=2)
                right_2 = plt.Polygon([[8.0, deep + 0.4], [8.0, 19.0]], color='black', linewidth=2)
                ax.add_patch(left)
                ax.add_patch(right_1)
                ax.add_patch(right_2)

                # 4. Parabolic water trajectory with noise
                y = np.linspace(deep, 0.5, num=50)
                epsilon = 0.01 * np.max([np.abs(np.random.randn()), 1.0])
                x = np.sqrt(2.0 * (0.98 + epsilon) * h * np.maximum(deep - y, 0.0)) + 8.0
                x_true = np.sqrt(2.0 * 0.98 * h * np.maximum(deep - 0.5, 0.0))

                ax.plot(x, y, color='lightskyblue', linewidth=2)

                # 5. Ground level
                x_ground = np.linspace(0.0, 20.0, num=50)
                y_ground = np.zeros(50) + 0.2
                ax.plot(x_ground, y_ground, color='black', linewidth=2)

                # Frame settings
                ax.set_xlim(0, 20)
                ax.set_ylim(0, 20)
                ax.axis('off')

                # Filename encodes full precision causal parameters to prevent overwriting
                # labels extracted: [r, h, x_true, hole]
                filename = f"a_{float(r):.2f}_{float(h):.2f}_{float(x_true):.2f}_{float(hole):.2f}.png"

                if count % test_split_ratio == (test_split_ratio - 1):
                    save_path = os.path.join(test_dir, filename)
                else:
                    save_path = os.path.join(train_dir, filename)

                fig.savefig(save_path, dpi=96, bbox_inches='tight', pad_inches=0)
                plt.close(fig)

                count += 1
                generated_count += 1

                if generated_count % 1000 == 0:
                    elapsed = time.time() - start_time
                    print(f"Rendered {generated_count}/{total_samples} images ({elapsed:.1f}s elapsed)")

    print(f"Finished generating {generated_count} causal samples in {time.time() - start_time:.2f}s.")


In [ ]:
# ==============================================================================
# Part 6: Training Pipeline & Optimization Loop (ICM-VAE)
# ==============================================================================

# ------------------------------------------------------------------------------
# Hyperparameters & Configurations
# ------------------------------------------------------------------------------
MAX_EPOCHS = 101
DATA = "flow"
SAVE_DIR = "icm_vae_recon"
NAME = "icm_vae_cdp"
DATASET_DIR = "./data/flow_noise"  # Matched with Part 5 output path
RUN = 0
TRAIN = 1
ITER_SAVE = 5
BATCH_SIZE = 64
LR = 1e-3

# Setup execution device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)

# Build experiment identifier
layout = [
    ("model={:s}", str(NAME)),
    ("run={:04d}", RUN),
    ("toy={:s}", str(DATA) + "_" + str(NAME)),
]
model_name = "_".join([t.format(v) for (t, v) in layout])
print(f"[*] Initialized Model Name: {model_name}")

# Prepare reconstruction output directory
recon_dir = f"./results/{DATA}/{DATA}_{NAME}_reconstructions/"
os.makedirs(recon_dir, exist_ok=True)

# Generate dataset automatically if not present
if not os.path.exists(os.path.join(DATASET_DIR, "train")):
    print(f"[*] Dataset not found at {DATASET_DIR}. Generating synthetic dataset...")
    generate_flow_dataset(output_dir=DATASET_DIR)


# ------------------------------------------------------------------------------
# Schedulers & Training Helpers
# ------------------------------------------------------------------------------
def linear_scheduler(step, total_steps, initial, final):
    """Linear annealing scheduler for regularizers."""
    if step >= total_steps:
        return final
    if step <= 0:
        return initial
    if total_steps <= 1:
        return final
    t = step / (total_steps - 1)
    return (1.0 - t) * initial + t * final


# ------------------------------------------------------------------------------
# Structural Matrices, Dataset & Model Initialization
# ------------------------------------------------------------------------------
# Adjacency matrix C
C = torch.tensor([[0, 0, 1, 0], [0, 0, 0, 1], [0, 0, 0, 1], [0, 0, 0, 0]], dtype=torch.float32)

# Scale parameter array (min, range)
scale = np.array([[20.0, 15.0], [10.5, 4.5], [2.0, 2.0], [59.5, 26.5]])

# Instantiate ICM_VAE model
icm_vae = ICM_VAE(
    name=f"{NAME}_{DATA}",
    z_dim=16,
    z1_dim=4,
    z2_dim=4,
    C=C,
    scale=scale,
).to(device)

# Dataset loader
train_dataset = get_batch_unin_dataset_withlabel(DATASET_DIR, BATCH_SIZE, dataset="train")
optimizer = torch.optim.Adam(icm_vae.parameters(), lr=LR, betas=(0.9, 0.999))

# ------------------------------------------------------------------------------
# Main Training Loop
# ------------------------------------------------------------------------------
print("[*] Starting training routine...")
num_batches = len(train_dataset)

for epoch in range(MAX_EPOCHS):
    icm_vae.train()
    total_loss = 0.0
    total_rec = 0.0
    total_kl = 0.0

    last_x = None
    last_recon = None

    for batch_idx, (X, l) in enumerate(train_dataset):
        X = X.to(device)
        l = l.to(device)

        optimizer.zero_grad()
        loss, kl, rec, reconstructed_image, z, cp_m = icm_vae.forward(
            X, l, sample=False
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_kl += kl.item()
        total_rec += rec.item()

        # Cache last sample from epoch for visualization
        last_x = X[0]
        last_recon = reconstructed_image[0]

    # Save representative reconstruction once per epoch
    if last_x is not None and last_recon is not None:
        save_image(last_x[:8], os.path.join(inference_dir, "true.png"))
        save_image(torch.sigmoid(last_recon[:8]), os.path.join(inference_dir, "reconstructed.png"))
    # Update annealing coefficients
    icm_vae.beta = linear_scheduler(epoch, 94, 0.0, 1.2)
    icm_vae.alpha = linear_scheduler(epoch, 94, 0.0, 0.1)

    # Logging
    avg_loss = total_loss / max(num_batches, 1)
    avg_kl = total_kl / max(num_batches, 1)
    avg_rec = total_rec / max(num_batches, 1)

    print(
        f"Epoch [{epoch:03d}/{MAX_EPOCHS:03d}] | "
        f"Loss: {avg_loss:.4f} | "
        f"KL: {avg_kl:.4f} | "
        f"Rec: {avg_rec:.4f} | "
        f"Beta: {icm_vae.beta:.3f} | "
        f"Alpha: {icm_vae.alpha:.3f}"
    )

    # Checkpoint saving
    if epoch % ITER_SAVE == 0:
        save_model_by_name(icm_vae, epoch)

print("[*] Training routine completed successfully.")


In [ ]:
# ==============================================================================
# Final Part: Inference, Disentanglement Evaluation & Metrics (DCI, IRS)
# ==============================================================================

# ------------------------------------------------------------------------------
# Configurations & Setup
# ------------------------------------------------------------------------------
DATA = "flow"
SAVE_DIR = "icm_vae_recon"
NAME = "icm_vae_cdp"
DATASET_DIR = "./data/flow_noise"  # Matched with Parts 5 & 6
RUN = 0
EVAL_CHECKPOINT = 100
BATCH_SIZE = 64

# Use unified device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model identifier
layout = [
    ("model={:s}", str(NAME)),
    ("run={:04d}", RUN),
    ("toy={:s}", str(DATA) + "_" + str(NAME)),
]
model_name = "_".join([t.format(v) for (t, v) in layout])
print(f"[*] Running Evaluation for: {model_name}")

# Prepare inference output directory
inference_dir = f"./results/{DATA}/{DATA}_{NAME}_inference/"
os.makedirs(inference_dir, exist_ok=True)

# ------------------------------------------------------------------------------
# Model Initialization & Checkpoint Loading
# ------------------------------------------------------------------------------
C = torch.tensor([[0, 0, 1, 0], [0, 0, 0, 1], [0, 0, 0, 1], [0, 0, 0, 0]], dtype=torch.float32)
scale = np.array([[20.0, 15.0], [10.5, 4.5], [2.0, 2.0], [59.5, 26.5]])

icm_vae = ICM_VAE(
    name=f"{NAME}_{DATA}",
    z_dim=16,
    z1_dim=4,
    z2_dim=4,
    C=C,
    scale=scale,
).to(device)

# Load weights
load_model_by_name(icm_vae, EVAL_CHECKPOINT)
icm_vae.eval()

# ------------------------------------------------------------------------------
# Dataset Loading
# ------------------------------------------------------------------------------
train_dataset = get_batch_unin_dataset_withlabel(DATASET_DIR, BATCH_SIZE, dataset="train")
test_dataset = get_batch_unin_dataset_withlabel(DATASET_DIR, BATCH_SIZE, dataset="test")

# ------------------------------------------------------------------------------
# Latent Extraction (Train & Test Split)
# ------------------------------------------------------------------------------
print("[*] Extracting latent representations from train split...")
rep_train_list, y_train_list = [], []

with torch.no_grad():
    for X, u in train_dataset:
        X = X.to(device)
        u = u.to(device)
        _, _, _, _, z, _ = icm_vae.forward(X, u, sample=False)
        
        rep_train_list.append(z.reshape(z.size(0), -1).cpu().numpy())
        y_train_list.append(u.reshape(u.size(0), -1).cpu().numpy())

rep_train = np.concatenate(rep_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

print("[*] Extracting latent representations from test split...")
rep_test_list, y_test_list = [], []
last_x, last_recon = None, None

with torch.no_grad():
    for X, u in test_dataset:
        X = X.to(device)
        u = u.to(device)
        _, _, _, reconstructed_image, z, _ = icm_vae.forward(X, u, sample=False)
        
        rep_test_list.append(z.reshape(z.size(0), -1).cpu().numpy())
        y_test_list.append(u.reshape(u.size(0), -1).cpu().numpy())
        
        last_x = X
        last_recon = reconstructed_image

rep_test = np.concatenate(rep_test_list, axis=0)
y_test = np.concatenate(y_test_list, axis=0)

# Save visual reconstructions of the final test batch
if last_x is not None and last_recon is not None:
    save_image(last_x[:8], os.path.join(inference_dir, "true.png"))
    save_image(last_recon[:8], os.path.join(inference_dir, "reconstructed.png"))
    print(f"[*] Saved test reconstructions to: {inference_dir}")

# ------------------------------------------------------------------------------
# Metric Computation
# ------------------------------------------------------------------------------
print("[*] Computing Disentanglement, Completeness, and Informativeness (DCI)...")
scores, importance_matrix, code_importance = _compute_dci(
    rep_train.T, y_train.T, rep_test.T, y_test.T
)

print("[*] Computing Interventional Robustness Score (IRS)...")
irs_score = compute_irs(rep_train.T, y_train.T)

# ------------------------------------------------------------------------------
# Results Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 50)
print("              EVALUATION RESULTS                 ")
print("=" * 50)
for k, v in scores.items():
    print(f"  {k:<25}: {v:.4f}")
print(f"  {'IRS':<25}: {irs_score['IRS']:.4f}")
print("-" * 50)
print("Importance Matrix (Codes x Factors):\n", np.round(importance_matrix, 4))
print("=" * 50)
